### Exercise 5 (c) — Python with pgmpy implementation
This notebook builds the Bayesian Network for campus lab cancellations using `pgmpy` to verify our hand-calculated results through both exact and approximate inference.


In [3]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling
from pgmpy.factors.discrete import State

# 1. Build the network structure using DiscreteBayesianNetwork
model = DiscreteBayesianNetwork([
    ('heatwave', 'overload'),
    ('heatwave', 'powercut'),
    ('overload', 'powercut'),
    ('powercut', 'cancel')
])

# Define Conditional Probability Tables (CPDs)
cpd_h = TabularCPD(
    variable='heatwave', variable_card=2, 
    values=[[0.3], [0.7]], 
    state_names={'heatwave': ['yes', 'no']}
)

cpd_a = TabularCPD(
    variable='overload', variable_card=2, 
    values=[[0.6, 0.1], 
            [0.4, 0.9]], 
    evidence=['heatwave'], evidence_card=[2],
    state_names={'overload': ['yes', 'no'], 'heatwave': ['yes', 'no']}
)

cpd_c = TabularCPD(
    variable='powercut', variable_card=2, 
    values=[[0.7, 0.2, 0.5, 0.05], 
            [0.3, 0.8, 0.5, 0.95]], 
    evidence=['heatwave', 'overload'], evidence_card=[2, 2],
    state_names={'powercut': ['yes', 'no'], 'heatwave': ['yes', 'no'], 'overload': ['yes', 'no']}
)

cpd_l = TabularCPD(
    variable='cancel', variable_card=2, 
    values=[[0.8, 0.1], 
            [0.2, 0.9]], 
    evidence=['powercut'], evidence_card=[2],
    state_names={'cancel': ['yes', 'no'], 'powercut': ['yes', 'no']}
)

# Attach CPDs to model and validate structure
model.add_cpds(cpd_h, cpd_a, cpd_c, cpd_l)
print(f"Model Check Status: {model.check_model()}")

# 2. Exact Inference using Variable Elimination
inference = VariableElimination(model)
exact_result = inference.query(variables=['heatwave'], evidence={'cancel': 'yes'})
print("\n--- Exact Inference Distribution ---")
print(exact_result)

# 3. Approximate Inference via Rejection Sampling
sampler = BayesianModelSampling(model)
samples = sampler.rejection_sample(evidence=[State("cancel", "yes")], size=10000, show_progress=False)
estimated_prob = (samples['heatwave'] == 'yes').mean()

print("\n--- Rejection Sampling Estimate ---")
print(f"P(heatwave = yes | cancel = yes) ≈ {estimated_prob:.4f}")


Model Check Status: True

--- Exact Inference Distribution ---
+---------------+-----------------+
| heatwave      |   phi(heatwave) |
+===============+=================+
| heatwave(yes) |          0.5367 |
+---------------+-----------------+
| heatwave(no)  |          0.4633 |
+---------------+-----------------+

--- Rejection Sampling Estimate ---
P(heatwave = yes | cancel = yes) ≈ 0.5411


#### 4. Comparison Analysis
The exact inference computed via Variable Elimination matches our hand calculations perfectly at **0.5367**. In contrast, the rejection sampling value will vary slightly with each execution loop (typically landing near ~0.53) because it approximates the underlying parameters through stochastic frequency instead of analytical equation expansions. By scaling our framework configuration to 10,000 trials, the sample population securely matches the actual math via the Law of Large Numbers.
